<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-4-fine-tuning/Module_4_Session_3_Base_vs_Finetuned_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 — Session 3: Base vs Fine-Tuned Model Evaluation
## Goal: Compare the base model and our fine-tuned model side by side using LLM-as-Judge scoring

In [1]:
# Install required libraries
!pip install unsloth groq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 703.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
from groq import Groq
from google.colab import userdata

# Load API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

print("Groq client ready!")

Groq client ready!


In [3]:
from unsloth import FastLanguageModel

# Load the base model — no LoRA, no fine-tuning
# This is our baseline — the model before any Swiggy training
model_base, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True
)

print("Base model loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded!


In [4]:
# 5 test complaints — neither model has seen these before
# We will send the exact same questions to both models and compare responses

test_complaints = [
    "My order from Bengaluru restaurant arrived 90 minutes late and the food was cold!",
    "I was charged twice for the same order. Please refund my money immediately.",
    "The delivery partner was speaking rudely and was very unprofessional.",
    "I ordered a veg meal but received non-veg food. I am a strict vegetarian!",
    "My promo code SWIGGY50 is not working. It says invalid but I just received it."
]

print(f"Test complaints ready: {len(test_complaints)}")

Test complaints ready: 5


In [5]:
# Alpaca prompt template — same as Session 2
alpaca_prompt = """Below is an instruction with an input. Write a response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def get_model_response(model, tokenizer, complaint):
    """
    Get a response from any model for a given complaint.

    Input: model, tokenizer, complaint string
    Output: response string
    """
    # Format the complaint using our template
    prompt = alpaca_prompt.format(
        "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        complaint,
        ""  # empty — model fills this in
    )

    # Tokenize and move to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=150)

    # Decode and extract only the response part
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full_response.split("### Response:")[-1].strip()

    return response

print("Response function ready!")

Response function ready!


In [6]:
# Switch base model to inference mode
FastLanguageModel.for_inference(model_base)

# Get responses from base model for all 5 complaints
print("Getting responses from BASE model...")
base_responses = []

for i, complaint in enumerate(test_complaints):
    print(f"  Complaint {i+1}/5...")
    response = get_model_response(model_base, tokenizer, complaint)
    base_responses.append(response)

print("\nBase model responses collected!")
print(f"Total: {len(base_responses)}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Getting responses from BASE model...
  Complaint 1/5...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  Complaint 2/5...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  Complaint 4/5...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Complaint 5/5...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



Base model responses collected!
Total: 5


In [7]:
import json
import random
from groq import Groq

# Recreate seed examples
seed_examples = [
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "It's been over an hour and my order hasn't arrived. I'm very hungry!",
        "output": "I sincerely apologize for this delay! I've tracked your order and see the delivery partner is 8 minutes away due to heavy rain. I've added ₹75 to your Swiggy wallet for this inconvenience. We value your patience!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "I ordered a Veg Burger but received a Chicken Burger. I am vegetarian!",
        "output": "I'm truly sorry for this serious mistake! I've raised a complaint with the restaurant and initiated a full refund of ₹249 within 5-7 business days. We take dietary requirements very seriously."
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order was cancelled automatically but money was deducted from my account.",
        "output": "I apologize for this inconvenience! Your payment of ₹380 will be refunded within 5-7 business days. I've added ₹50 to your Swiggy wallet to compensate. Please feel free to reorder!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order arrived but the french fries were missing. I paid ₹80 extra for them!",
        "output": "I'm sorry about the missing fries! I've verified your order and refunded ₹80 to your Swiggy wallet immediately. Thank you for flagging this!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "The delivery partner was very rude to me and threw the food bag at my door.",
        "output": "I sincerely apologize for this unacceptable behavior! I've flagged this for immediate review. I've added ₹150 to your Swiggy wallet as compensation. You deserve respectful service always!"
    }
]

print(f"Seed examples ready: {len(seed_examples)}")

Seed examples ready: 5


In [8]:
# Generate synthetic examples and save as JSONL
categories = ["order_delay", "wrong_item", "refund_request",
              "missing_item", "app_issue", "delivery_behavior"]

def generate_examples_for_category(category, n=4):
    prompt = f"""You are creating training data for a Swiggy customer support AI.

Generate exactly {n} different customer support examples for the category: {category}

Rules:
- instruction must always be: "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution."
- input must be a realistic Indian customer complaint (1-2 sentences)
- output must be an empathetic agent response (3-4 sentences, always offer a solution)
- Use Indian context: mention ₹ amounts, Indian cities like Mumbai/Delhi/Bengaluru/Chennai

Return ONLY a valid JSON array, no explanation, no markdown, no extra text.
Format:
[
  {{"instruction": "...", "input": "...", "output": "..."}},
  {{"instruction": "...", "input": "...", "output": "..."}}
]"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8
    )
    return json.loads(response.choices[0].message.content)

# Generate for all categories
all_examples = seed_examples.copy()
for category in categories:
    print(f"Generating: {category}...")
    try:
        new_examples = generate_examples_for_category(category, n=4)
        all_examples.extend(new_examples)
    except Exception as e:
        print(f"  Error: {e}")

# Shuffle and save
random.seed(42)
random.shuffle(all_examples)
train_data = all_examples[:int(len(all_examples) * 0.8)]

with open("swiggy_train.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print(f"\nTotal examples: {len(all_examples)}")
print(f"Train examples saved: {len(train_data)}")

Generating: order_delay...
Generating: wrong_item...
Generating: refund_request...
Generating: missing_item...
Generating: app_issue...
Generating: delivery_behavior...

Total examples: 29
Train examples saved: 23


In [9]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# Step 1 — Attach LoRA adapters to base model
model_finetuned = FastLanguageModel.get_peft_model(
    model_base,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# Step 2 — Load and format dataset
train_data_loaded = []
with open("swiggy_train.jsonl", "r") as f:
    for line in f:
        train_data_loaded.append(json.loads(line.strip()))

formatted_texts = []
for ex in train_data_loaded:
    text = alpaca_prompt.format(
        ex["instruction"], ex["input"], ex["output"]
    ) + tokenizer.eos_token
    formatted_texts.append({"text": text})

train_dataset = Dataset.from_list(formatted_texts)

# Step 3 — Train
trainer = SFTTrainer(
    model = model_finetuned,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1,
        save_strategy = "no",
    ),
)

trainer.train()
print("Fine-tuning complete!")

Unsloth 2026.6.9 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/23 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 23 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.451248
2,2.680867
3,2.180868
4,2.127970
5,1.855669
6,1.592343
7,1.788528
8,1.466018
9,1.143213
10,1.316811


Fine-tuning complete!


In [10]:
# Switch fine-tuned model to inference mode
FastLanguageModel.for_inference(model_finetuned)

# Get responses from fine-tuned model for all 5 complaints
print("Getting responses from FINE-TUNED model...")
finetuned_responses = []

for i, complaint in enumerate(test_complaints):
    print(f"  Complaint {i+1}/5...")
    response = get_model_response(model_finetuned, tokenizer, complaint)
    finetuned_responses.append(response)

print("\nFine-tuned model responses collected!")
print(f"Total: {len(finetuned_responses)}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Getting responses from FINE-TUNED model...
  Complaint 1/5...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Complaint 2/5...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Complaint 3/5...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Complaint 4/5...


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Complaint 5/5...

Fine-tuned model responses collected!
Total: 5


In [11]:
import pandas as pd

# Print both responses side by side for each complaint
for i in range(5):
    print(f"\n{'='*60}")
    print(f"COMPLAINT {i+1}: {test_complaints[i]}")
    print(f"\nBASE MODEL:")
    print(base_responses[i])
    print(f"\nFINE-TUNED MODEL:")
    print(finetuned_responses[i])


COMPLAINT 1: My order from Bengaluru restaurant arrived 90 minutes late and the food was cold!

BASE MODEL:
"Hello there! Sorry to hear that your order from our Bengaluru restaurant arrived late and the food was cold. I'm here to help you resolve this issue. Can you please provide me with your order number so I can look into this further? Additionally, I'll make sure to expedite the delivery and provide you with a refund for the food that was delivered late. Please let me know if there's anything else I can assist you with. We value your business and want to ensure that you're satisfied with your experience on Swiggy."

FINE-TUNED MODEL:
I'm so sorry to hear that your order was delayed and the food was cold. I'd like to offer you a 20% discount on your next order as a gesture of goodwill. Please let me know your order number so I can look into this further.

COMPLAINT 2: I was charged twice for the same order. Please refund my money immediately.

BASE MODEL:
Thank you for reaching out

In [12]:
def score_response(complaint, response):
    """
    Use Groq LLM as a judge to score a support response.

    Input: complaint string, response string
    Output: dictionary with scores and average
    """
    prompt = f"""You are evaluating a customer support response for Swiggy food delivery app.

Customer complaint: {complaint}
Support response: {response}

Score the response on these 3 criteria from 1-10:
1. Empathy — does it acknowledge the customer's frustration?
2. Solution — does it offer a concrete fix or compensation?
3. Tone — does it sound professional and helpful?

Return ONLY a valid JSON object, no explanation, no markdown.
Format:
{{"empathy": 7, "solution": 8, "tone": 9, "average": 8.0}}"""

    result = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    # Parse JSON response
    scores = json.loads(result.choices[0].message.content)
    return scores

# Test with first complaint
print("Testing judge on complaint 1...")
test_score = score_response(test_complaints[0], base_responses[0])
print(test_score)

Testing judge on complaint 1...
{'empathy': 7, 'solution': 8, 'tone': 9, 'average': 8.0}


In [13]:
# Score all 5 complaints for both models
print("Scoring all responses...\n")

base_scores = []
finetuned_scores = []

for i in range(5):
    print(f"Scoring complaint {i+1}/5...")

    # Score base model response
    base_score = score_response(test_complaints[i], base_responses[i])
    base_scores.append(base_score)

    # Score fine-tuned model response
    finetuned_score = score_response(test_complaints[i], finetuned_responses[i])
    finetuned_scores.append(finetuned_score)

print("\nAll scoring complete!")

Scoring all responses...

Scoring complaint 1/5...
Scoring complaint 2/5...
Scoring complaint 3/5...
Scoring complaint 4/5...
Scoring complaint 5/5...

All scoring complete!


In [14]:
# Build comparison DataFrame
results = []

for i in range(5):
    results.append({
        "complaint"         : test_complaints[i][:50] + "...",  # truncate for display
        "base_empathy"      : base_scores[i]["empathy"],
        "base_solution"     : base_scores[i]["solution"],
        "base_tone"         : base_scores[i]["tone"],
        "base_average"      : base_scores[i]["average"],
        "ft_empathy"        : finetuned_scores[i]["empathy"],
        "ft_solution"       : finetuned_scores[i]["solution"],
        "ft_tone"           : finetuned_scores[i]["tone"],
        "ft_average"        : finetuned_scores[i]["average"],
    })

df = pd.DataFrame(results)

# Print summary
print("="*60)
print("FINAL SCORES — BASE vs FINE-TUNED")
print("="*60)
print(df[["complaint", "base_average", "ft_average"]])

print("\n")
print(f"Base model average score     : {df['base_average'].mean():.2f}")
print(f"Fine-tuned model average score: {df['ft_average'].mean():.2f}")

FINAL SCORES — BASE vs FINE-TUNED
                                           complaint  base_average  ft_average
0  My order from Bengaluru restaurant arrived 90 ...         8.000         8.0
1  I was charged twice for the same order. Please...         8.000         8.0
2  The delivery partner was speaking rudely and w...         8.000         8.0
3  I ordered a veg meal but received non-veg food...         8.000         8.0
4  My promo code SWIGGY50 is not working. It says...         7.333         8.0


Base model average score     : 7.87
Fine-tuned model average score: 8.00
